# EDA — RACE Dataset
Exploratory Data Analysis covering passage lengths, question types, answer balance, word frequency, and preprocessing verification.

In [ ]:
import sys, os
sys.path.insert(0, '../src')
os.chdir('..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from collections import Counter
from preprocessing import load_race_dataset, tokenize, remove_stopwords, clean_text, split_sentences
plt.style.use('dark_background')
print('Imports OK')

## 1. Load Dataset

In [ ]:
train_df, val_df, test_df = load_race_dataset('data/raw')
print(f'Train: {train_df.shape}  Val: {val_df.shape}  Test: {test_df.shape}')
train_df.head(3)

## 2. Summary Statistics

In [ ]:
train_df['article_len']  = train_df['article'].apply(lambda x: len(str(x).split()))
train_df['question_len'] = train_df['question'].apply(lambda x: len(str(x).split()))
train_df['answer_len']   = train_df.apply(lambda r: len(str(r[r['answer']]).split()), axis=1)
print(train_df[['article_len','question_len','answer_len']].describe().round(2))

## 3. Passage Length Distribution

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,4))
colors = ['#4A90D9','#E87040','#50C878']
cols   = ['article_len','question_len','answer_len']
titles = ['Article Length (words)','Question Length (words)','Answer Length (words)']
for ax, col, title, color in zip(axes, cols, titles, colors):
    ax.hist(train_df[col], bins=40, color=color, alpha=0.85)
    ax.set_title(title); ax.set_xlabel('Word Count'); ax.set_ylabel('Frequency')
plt.tight_layout()
os.makedirs('data/processed', exist_ok=True)
plt.savefig('data/processed/eda_length_dist.png', dpi=100)
plt.show()

## 4. Answer Balance (A/B/C/D)

In [ ]:
ac = train_df['answer'].value_counts().sort_index()
print('Answer Distribution:'); print(ac)
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(ac.index, ac.values, color=['#4A90D9','#E87040','#50C878','#9B59B6'])
ax.set_title('Answer Label Distribution'); ax.set_xlabel('Label'); ax.set_ylabel('Count')
plt.tight_layout(); plt.savefig('data/processed/eda_answer_balance.png', dpi=100); plt.show()

## 5. Question Type Analysis (Wh-words)

In [ ]:
wh_words = ['what','why','who','how','when','where','which']
wh_counts = {wh:0 for wh in wh_words}; wh_counts['other']=0
for q in train_df['question']:
    q_l = str(q).lower(); found=False
    for wh in wh_words:
        if q_l.startswith(wh): wh_counts[wh]+=1; found=True; break
    if not found: wh_counts['other']+=1
wh_s = pd.Series(wh_counts).sort_values(ascending=False)
print(wh_s)
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(wh_s.index, wh_s.values, color='#4A90D9')
ax.set_title('Question Types'); ax.set_xlabel('Wh-word'); ax.set_ylabel('Count')
plt.tight_layout(); plt.savefig('data/processed/eda_question_types.png', dpi=100); plt.show()

## 6. Top Frequent Words

In [ ]:
all_tokens=[]
for art in train_df['article'].head(500): all_tokens.extend(remove_stopwords(tokenize(str(art))))
freq=Counter(all_tokens); top=freq.most_common(20)
words,counts=zip(*top)
fig, ax = plt.subplots(figsize=(10,5))
ax.barh(list(words)[::-1], list(counts)[::-1], color='#E87040')
ax.set_title('Top 20 Frequent Words (stopwords removed)')
plt.tight_layout(); plt.savefig('data/processed/eda_top_words.png', dpi=100); plt.show()

## 7. Preprocessing Verification

In [ ]:
sample = train_df['article'].iloc[0]
print('ORIGINAL:'); print(sample[:200])
print('\nCLEANED:'); print(clean_text(sample[:200]))
print('\nTOKENIZED (first 15):', tokenize(sample[:200])[:15])
print('\nNO STOPWORDS (first 15):', remove_stopwords(tokenize(sample[:200]))[:15])

## 8. Summary Table

In [ ]:
summary = pd.DataFrame({
    'Metric':['Train Rows','Val Rows','Test Rows','Avg Article Len','Avg Question Len','Avg Answer Len'],
    'Value':[len(train_df),len(val_df),len(test_df),
             f"{train_df['article_len'].mean():.1f}",
             f"{train_df['question_len'].mean():.1f}",
             f"{train_df['answer_len'].mean():.1f}"]
})
print(summary.to_string(index=False))